# MSD Task09 Spleen: 3D U-Net Model Training

> **Educational prototype only. Not for clinical diagnostic use.**

This notebook executes the **training stage** with all architectural and algorithmic fixes:
1. **Decoupled dynamic crop pipeline**: Caches deterministic volumes in RAM and extracts 4 fresh positive/negative crops per volume per epoch.
2. **Instance Normalization (`Norm.INSTANCE`)**: Eliminates batch norm collapse for batch size 1.
3. **Prior Probability Bias Initialization (`prior_prob=0.01`)**: Solves the initial sigmoid saturation defect that caused 85x over-prediction.
4. **Balanced channel hierarchy**: `channels=(16, 32, 64, 128)` and `strides=(2, 2, 2)`.
5. **Compound `DiceCELoss`**: Directly penalizes false-positive background voxels.
6. **Mixed-Precision GPU training with Gradient Clipping**: Safe and efficient execution on RTX 3050 Laptop GPU.
7. **Checkpointing**: Automatically saves the best model based on held-out validation Dice.

In [ ]:
import os
import sys
import time
from pathlib import Path
# Locate project root and ensure it is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import torch
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

import monai
from monai.data import CacheDataset, Dataset, DataLoader, list_data_collate
from monai.inferers import sliding_window_inference
from monai.losses import DiceCELoss
from monai.transforms import Compose, RandCropByPosNegLabeld, RandRotated, RandFlipd

from ml.segmentation.model import create_spleen_unet, dice_score, post_process_mask
from ml.segmentation.preprocessing import (
    IMAGE_KEY,
    LABEL_KEY,
    build_preprocessing_transforms,
    list_volume_pairs,
)

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parents[1]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Training device:', device)
if device.type == 'cuda':
    print('GPU Model:', torch.cuda.get_device_name(0))
    print('CUDA Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

In [ ]:
config_path = project_root / 'ml' / 'segmentation' / 'config.yaml'
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))

# Training scope for prototype training
TRAIN_VOLUME_COUNT = 16
VAL_VOLUME_COUNT = 4
EPOCHS = 40
LEARNING_RATE = 2e-4
ROI_SIZE = (96, 96, 96)
NUM_CROPS_PER_VOLUME = 4

images_dir = project_root / config['dataset']['images_dir']
labels_dir = project_root / config['dataset']['labels_dir']
all_pairs = list_volume_pairs(images_dir, labels_dir)

train_pairs = all_pairs[:TRAIN_VOLUME_COUNT]
val_pairs = all_pairs[TRAIN_VOLUME_COUNT : TRAIN_VOLUME_COUNT + VAL_VOLUME_COUNT]
print(f'Train volumes: {len(train_pairs)}, Validation volumes: {len(val_pairs)}')

In [ ]:
# 1. Deterministic preprocessing cached in RAM
prep_transforms = build_preprocessing_transforms(config)

print('Caching training volumes in memory...')
train_cache = CacheDataset(data=train_pairs, transform=prep_transforms, cache_rate=1.0, num_workers=0)

print('Caching validation volumes in memory...')
val_cache = CacheDataset(data=val_pairs, transform=prep_transforms, cache_rate=1.0, num_workers=0)

# 2. Dynamic random cropping and spatial augmentation applied on-the-fly
dynamic_crop_transforms = Compose([
    RandCropByPosNegLabeld(
        keys=[IMAGE_KEY, LABEL_KEY],
        label_key=LABEL_KEY,
        spatial_size=ROI_SIZE,
        pos=2.0,
        neg=1.0,
        num_samples=NUM_CROPS_PER_VOLUME,
        image_key=IMAGE_KEY,
        image_threshold=0.0,
    ),
    RandFlipd(keys=[IMAGE_KEY, LABEL_KEY], prob=0.5, spatial_axis=0),
    RandFlipd(keys=[IMAGE_KEY, LABEL_KEY], prob=0.5, spatial_axis=1),
    RandRotated(keys=[IMAGE_KEY, LABEL_KEY], range_x=0.2, prob=0.3, mode=('bilinear', 'nearest')),
])

train_dataset = Dataset(data=train_cache, transform=dynamic_crop_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=list_data_collate,
    pin_memory=(device.type == 'cuda'),
)

val_loader = DataLoader(
    val_cache,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)
print(f'Train batches per epoch: {len(train_loader)} (total {len(train_loader)*NUM_CROPS_PER_VOLUME} crops/epoch)')

In [ ]:
# Instantiate 3D U-Net with Norm.INSTANCE and prior probability initialization (prior_prob=0.01)
model = create_spleen_unet(prior_prob=0.01).to(device)

# Compound Dice + Cross-Entropy Loss
loss_function = DiceCELoss(sigmoid=True, lambda_dice=1.0, lambda_ce=1.0)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

print('Model architecture successfully built with Instance Normalization & prior_prob=0.01.')
print('Loss function: Compound DiceCELoss(lambda_dice=1.0, lambda_ce=1.0)')

In [ ]:
models_dir = project_root / 'models' / 'segmentation'
models_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = models_dir / 'best_spleen_unet.pt'

best_val_dice = -1.0
best_epoch = -1
history = []

print(f'Starting training for {EPOCHS} epochs...')
overall_start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    model.train()
    train_loss = 0.0
    step_count = 0

    for batch in train_loader:
        images = batch[IMAGE_KEY].to(device, non_blocking=True)
        labels = batch[LABEL_KEY].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = loss_function(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += float(loss.detach().cpu())
        step_count += 1

    scheduler.step()
    avg_train_loss = train_loss / max(step_count, 1)

    # Validation step with whole-volume sliding window inference
    do_val = (epoch % 2 == 0 or epoch == 1 or epoch == EPOCHS)
    val_dices = []
    if do_val:
        model.eval()
        with torch.no_grad():
            for val_batch in val_loader:
                val_img = val_batch[IMAGE_KEY].to(device)
                val_lbl = val_batch[LABEL_KEY].to(device)

                with torch.autocast(device_type=device.type, enabled=(device.type == 'cuda')):
                    val_logits = sliding_window_inference(
                        inputs=val_img,
                        roi_size=ROI_SIZE,
                        sw_batch_size=1,
                        predictor=model,
                        overlap=0.5,
                        mode='gaussian',
                    )
                val_pred = post_process_mask(val_logits, threshold=0.5, keep_largest_component=True)
                val_dices.append(dice_score(val_pred, val_lbl))

        mean_val_dice = float(np.mean(val_dices))
    else:
        mean_val_dice = history[-1]['val_dice'] if history else 0.0

    epoch_time = time.perf_counter() - epoch_start

    history.append({
        'epoch': epoch,
        'train_loss': avg_train_loss,
        'val_dice': mean_val_dice,
        'time_sec': epoch_time,
    })

    saved_msg = ''
    if do_val and mean_val_dice > best_val_dice:
        best_val_dice = mean_val_dice
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'validation_dice': mean_val_dice,
            'loss': 'dice_ce',
            'train_volume_count': len(train_pairs),
            'validation_volume_count': len(val_pairs),
            'roi_size': ROI_SIZE,
            'norm': 'instance',
        }, checkpoint_path)
        saved_msg = ' ==> [SAVED BEST CHECKPOINT]'

    val_str = f'{mean_val_dice:.4f}' if do_val else f'({mean_val_dice:.4f} cached)'
    print(f'Epoch {epoch:02d}/{EPOCHS:02d} | Train Loss: {avg_train_loss:.4f} | ' 
          f'Val Dice: {val_str} | Time: {epoch_time:.1f}s{saved_msg}')

total_time = time.perf_counter() - overall_start
print(f'\nTraining completed in {total_time:.1f}s.')
print(f'Best Validation Dice: {best_val_dice:.4f} at Epoch {best_epoch}')
print(f'Saved to: {checkpoint_path}')

In [ ]:
# Plot training loss and validation Dice progression
epochs_arr = [h['epoch'] for h in history]
losses_arr = [h['train_loss'] for h in history]
dices_arr = [h['val_dice'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_arr, losses_arr, 'o-', color='crimson', label='DiceCE Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(epochs_arr, dices_arr, 's-', color='teal', label='Held-out Dice')
ax2.axhline(0.70, color='gray', linestyle='--', label='Prototype Target (0.70)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Dice Score')
ax2.set_title('Validation Dice')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()